### 구현할 기능 목록
* 메일 확인 (기간 & 키워드 조건)
* 메일 전송 기능
* (+ 메일 템플릿 추천 -> 에이전트와 결합할 때, LLM Agent 연결을 통해 구현?)
* (+ Query 프롬프트로부터 각 메서드의 인수들을, LLM Agent를 통해 적절한 type으로 받을 수 있도록 )

In [39]:
import os.path
import base64
import datetime

# Google 라이브러리 import
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

from bs4 import BeautifulSoup


# 이메일 작성을 위한 라이브러리
from email.mime.text import MIMEText

In [2]:
# GMAIL의 권한 설정
SCOPES = [
    "https://www.googleapis.com/auth/gmail.readonly", # 읽기
    "https://www.googleapis.com/auth/gmail.send"      # 보내기
]

In [ ]:
# Google API 서비스 인증


def get_gamail_service():
    '''
    Google Gmail API 서비스 객체를 인증 및 반환.
    token.json이 경로 내에 존재하지 않으면 새로 생성.
    '''

    creds = None

    # 'token.json' : 인증 정보를 저장
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)

    # 토큰이 없거나 유효하지 않은 경우, 사용자가 직접 로그인하도록 함
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                'google_oauth_credentials.json', SCOPES
            )
            creds = flow.run_local_server(port=0)

    # 다음 번 실행을 위해 토큰을 저장
    with open('token.json', 'w') as token:
        token.write(creds.to_json())

    # API 서비스 빌드
    try:
        service = build('gmail', 'v1', credentials=creds)
        
        # 로그
        print("Gmail 서비스 인증 성공")
        
        return service
    
    except HttpError as e:
        # 로그
        print(f"서비스 빌드 실패 : {e}")

        return None

In [61]:
def list_emails_by_keyword_and_date(service, start_date, end_date, keyword, message_num = 10):
    '''
    'start_date' ~ 'end_date'에 수신한 메일 중 메일 '제목' or '내용'에 특정 키워드가 포함된 메일 제목 확인
    '''

    email_list = []

    try:
        # 1.1. KST 기준 시작일, 종료일 설정
        KST = datetime.timezone(datetime.timedelta(hours=9))
        date_start_kst = datetime.datetime.strptime(start_date, '%Y-%m-%d').replace(tzinfo=KST)
        date_end_kst =  datetime.datetime.strptime(end_date, '%Y-%m-%d').replace(tzinfo=KST) + datetime.timedelta(days=1)

        # 1.2. Unix 타임스탬프로 변환
        st_timestamp = int(date_start_kst.timestamp())
        end_timestamp = int(date_end_kst.timestamp())

        # 2. Gmail 검색 쿼리
        query = f"(subject:({keyword}) OR body:({keyword})) -in:sent after:{st_timestamp} before:{end_timestamp}"

        # 3. 메일 검색
        results = service.users().messages().list(userId='me', q=query).execute()
        messages = results.get('messages', [])

        if not messages:
            print("해당 키워드를 포함하는 메일이 존재하지 않음.")
            return []
        
        # 4. 각 메일 추출
        for msg in messages[:message_num]:
            msg_data = service.users().messages().get(
                userId='me', id=msg['id'], format='full'
            ).execute()

            # 4.1. 본문 추출
            payload = msg_data['payload']
            body_data = ""
            decoded_body = "본문 없음"

            # HTML, TEXT 등 여러 part로 나뉜 경우
            if 'parts' in payload:
                for part in payload['parts']:
                    if part['mimeType'] == 'text/plain':
                        body_data = part['body'].get('data', '')
                        break
                        
                    elif part['mimeType'] == 'text/html':
                        body_data = part['body'].get('data', '')

            # 단순 텍스트 메일인 경우
            elif 'body' in payload:
                body_data = payload['body'].get('data', '')

            if body_data:
                try:
                    decoded_html = base64.urlsafe_b64decode(body_data.encode('ASCII')).decode('utf-8')

                    soup = BeautifulSoup(decoded_html, 'html.parser')
                    decoded_body = soup.get_text(separator=' ', strip=True)
                
                except Exception as e:
                    decoded_body = f"본문 파싱 실패 : {e}"

            # 4.2. 제목 및 발신인 추출
            headers = payload['headers']

            subject = next((h['value'] for h in headers if h['name'] == 'Subject'), "제목 없음")
            sender = next((h['value'] for h in headers if h['name'] == 'From'), "발신인 불명")

            # 4.3. Gmail 링크 생성
            mail_id = msg['id']
            mail_link = f"https://mail.google.com/mail/u/1/#inbox/{mail_id}" # gmail 계정이 여러 개일 경우 계정의 순서에 맞게 0부터 순차적으로 증가하는 적절한 숫자를 '.../u/{숫자}/#inbox/...'의 {숫자}에 삽입하면 됨!
            
            email_list.append([subject, sender, decoded_body, mail_link])

        return email_list

    except HttpError as e:
        print(f"키워드 기준 메일 제목 확인 실패 : {e}")

        return []

In [26]:
gmail_service = get_gamail_service()

Gmail 서비스 인증 성공


In [62]:
results = list_emails_by_keyword_and_date(gmail_service, '2025-09-16', '2025-09-16', '출석', 5)

In [63]:
for 제목, 발신인, 본문, 링크 in results:
    print(f"Title: {제목}")
    print(f"From : {발신인}")
    print(f"Text : \n{본문}")
    print(f"GMAIL Link : {링크}")
    print("-"*100)

Title: Re: [조합최적화(001) 강좌 출석 관련 문의]
From : "노영주" <youngjooroh@snu.ac.kr>
Text : 
안녕하세요, 강의조교 노영주 입니다.

보내주신 서류 잘 확인했습니다.
해당 일자는 출석인정 처리하도록 하겠습니다.

감사합니다.

노영주 드림

*Youngjoo Roh*
Ph.D. Student
Systems Optimization Lab.
Department of Industrial Engineering, Seoul National University
*Email* youngjooroh@snu.ac.kr | youngjooroh.yjr@gmail.com


2025년 9월 16일 (화) 오후 12:20, ­임다움 / 학생 / 산업공학과 님이 작성:

> 안녕하세요, 조교님.
> 홍성필 교수님의 조합최적화(001) 강좌를 수강하고 있는 산업공학과에 재학 중인 임다움입니다.
>
> 다름이 아니라, 비염 및 편도염 증세가 심해져 병원에 방문하게 되어 부득이하게 금일 09.16(화)에 진행되었던 강의에 출석하지
> 못했습니다. 병원에서 처방전을 발급해주어 첨부하오니, 이를 통해 출석을 인정 받을 수 있는지 문의드리고자 메일 드립니다. 혹시, 이외의
> 증명을 위한 서류가 필요한 경우 말씀해주시면 추후 제출하도록 하겠습니다.
>
> 감사합니다.
>
> 임다움 드림
>
GMAIL Link : https://mail.google.com/mail/u/1/#inbox/19950cfa1ca452fb
----------------------------------------------------------------------------------------------------
